 # `aidweather` — Practical Guide

 **Author:** Cleverson Matiolli, PhD

 ---

 `aidweather` retrieves daily or hourly weather data from
 [NASA POWER](https://power.larc.nasa.gov/) and caches it locally so repeated
 requests don't hit the network again.

 This notebook walks through every public component of the package:

 | Section | What you'll learn |
 |---------|-------------------|
 | 1 · Configuration | Inspect API endpoints and available weather parameters |
 | 2 · Coordinates | Parse, validate and convert geographic coordinates |
 | 3 · Fetching data | Download weather data for a single location |
 | 4 · Caching | See how the cache speeds up repeated queries |
 | 5 · Multi-point | Fetch data for several locations in parallel |
 | 6 · Data cleanup | Standardise the date column in any DataFrame |

In [1]:
import time
from pathlib import Path

import pandas as pd

from aidweather import (
    GeoCoordinate,
    PowerClient,
    ensure_date_column,
    get_config,
    normalize_coord_input,
)

print("aidweather ready.")

aidweather ready.


 ---
 ## 1 · Configuration

 `cfg` is a module-level singleton that reads `assets/config.json` once and
 exposes helpers for API URLs and parameter metadata.
 `get_config()` returns the same object — both refer to the same instance.

In [2]:
config = get_config()

# API base URLs
print("Daily point URL  :", config.get_url("daily", "point"))
print("Hourly point URL :", config.get_url("hourly", "point"))

Daily point URL  : https://power.larc.nasa.gov/api/temporal/daily/point
Hourly point URL : https://power.larc.nasa.gov/api/temporal/hourly/point


 ### Available parameter groups

 Parameters are grouped in `config.json`.
 The `"default"` group is a curated set of variables useful for most agronomic work.

In [3]:
default_params = config.params("default")
print(f"Default group — {len(default_params)} parameters:\n")
for code, label in default_params.items():
    print(f"  {code:<22} {label}")

Default group — 7 parameters:

  T2M                    Temperature at 2 m (°C)
  T2M_RANGE              Temperature Range at 2 m (°C)
  T2MDEW                 Dew Point Temperature at 2 m (°C)
  RH2M                   Relative Humidity at 2 m (%)
  PRECTOTCORR            Corrected Total Precipitation (mm/day)
  ALLSKY_SFC_PAR_TOT     All Sky Photosynthetically Active Radiation (MJ/m²/day)
  WS10M                  Wind Speed at 10 m (m/s)


 The "all" group contains all available parameters.

In [4]:
all_params = config.params("all")
print(f"All group — {len(all_params)} parameters:\n")
for code, label in all_params.items():
    print(f"  {code:<22} {label}")

All group — 15 parameters:

  T2M                    Temperature at 2 m (°C)
  T2M_MAX                Maximum Temperature at 2 m (°C)
  T2M_MIN                Minimum Temperature at 2 m (°C)
  T2M_RANGE              Temperature Range at 2 m (°C)
  T2MWET                 Wet Bulb Temperature at 2 m (°C)
  T2MDEW                 Dew Point Temperature at 2 m (°C)
  TS                     Surface Skin Temperature (°C)
  RH2M                   Relative Humidity at 2 m (%)
  PRECTOTCORR            Corrected Total Precipitation (mm/day)
  ALLSKY_SFC_SW_DWN      All Sky Surface Shortwave Downward Radiation (kWh/m²/day)
  ALLSKY_SFC_PAR_TOT     All Sky Photosynthetically Active Radiation (MJ/m²/day)
  WS10M                  Wind Speed at 10 m (m/s)
  WS10M_MAX              Maximum Wind Speed at 10 m (m/s)
  WD10M                  Wind Direction at 10 m (degrees)
  PS                     Surface Pressure (kPa)


 ### Parameter descriptions

 Each parameter has a longer description stored in the config.

In [5]:
descriptions = config.param_descriptions()
for code in all_params:
    print(f"[{code}]\n  {descriptions.get(code, 'n/a')}\n")

[T2M]
  Average Air Temperature at 2 m (°C): Modeled daily average air temperature. Data is obtained from the NASA MERRA-2 assimilation model (1981-Present, ~0.5°x0.625° resolution). In agriculture, T2M is critical for calculating Growing Degree Days (GDD) and modeling crop developmental stages and growth rates. Used in machine learning models for yield prediction and thermal stress assessment.

[T2M_MAX]
  Maximum Air Temperature at 2 m (°C): Modeled maximum hourly air temperature. Data is obtained from the NASA MERRA-2 assimilation model (1981-Present, ~0.5°x0.625° resolution). T2M_MAX is vital for detecting heat stress events which can cause flower abortion, reduced photosynthesis, and poor grain filling. Used as a feature in ML models to flag high-temperature risk periods.

[T2M_MIN]
  Minimum Air Temperature at 2 m (°C): Modeled minimum hourly air temperature. Data is obtained from the NASA MERRA-2 assimilation model (1981-Present, ~0.5°x0.625° resolution). T2M_MIN is essential fo

 ---
 ## 2 · Coordinates

 `GeoCoordinate` accepts coordinates in **any common format** and converts
 between them.  It also validates ranges before you ever touch the API.

 Supported input styles:
 - Decimal degrees as a float: `-23.55`
 - Decimal degrees as a string: `"-23.55°"`
 - Degrees and decimal minutes: `"23° 33.0' S"`
 - Degrees, minutes, seconds: `"23° 33' 0\" S"`

In [6]:
# From two strings (any mix of formats)
c1 = GeoCoordinate.from_strings("23° 33.0' S", "46° 37' 48\" W")

# From two floats (south/west are negative)
c2 = GeoCoordinate.from_decimal(-23.55, -46.63)

for i, c in enumerate([c1, c2], 1):
    print(f"Coord {i}:  lat={c.lat:.5f}  lon={c.lon:.5f}")

Coord 1:  lat=-23.55000  lon=-46.63000
Coord 2:  lat=-23.55000  lon=-46.63000


 ### Mixed-format input via `normalize_coord_input`

 If you have a tuple where each element could be a float or a string,
 `normalize_coord_input` handles it in one call.

In [7]:
coord = normalize_coord_input((-23.55, "46°37'48.0\" W"))
print("Normalised:", coord)

Normalised: GeoCoordinate(lat=-23.55, lon=-46.63)


 ### Export to formatted strings

In [8]:
print("DD  :", coord.to_dd_str(lat_precision=5))
print("DDM :", coord.to_ddm_str(minute_precision=3))
print("DMS :", coord.to_dms_str(second_precision=2))

DD  : ('23.55000° S', '46.63000° W')
DDM : ("23°33.000' S", "46°37.800' W")
DMS : ('23°33\'0.00" S', '46°37\'48.00" W')


 ### Validation

 Invalid coordinates raise a `ValueError` immediately — before any API call is made.

In [9]:
try:
    GeoCoordinate.from_decimal(95.0, -46.63)  # latitude > 90 is invalid
except ValueError as e:
    print("Caught:", e)

Caught: Latitude out of range [-90, 90]: 95.0


 ---
 ## 3 · Fetching weather data

 `PowerClient` wraps the NASA POWER API.
 Initialise it once and reuse it across queries.

 ```python
 client = PowerClient(temporal_api="daily")   # or "hourly"
 ```

 ### `get_point_data` — single location

 Returns a **pandas DataFrame** indexed by date, one column per parameter.

In [10]:
client = PowerClient(temporal_api="daily")

lat, lon = -23.31, -51.16  # Londrina, PR, Brazil
start, end = "2023-01-01", "2023-01-15"
params = ["T2M", "PRECTOTCORR", "ALLSKY_SFC_PAR_TOT"]

df = client.get_point_data(lat=lat, lon=lon, start=start, end=end, params=params)

print(df.shape)  # (days, parameters)
print(df.head())

(15, 3)
              T2M  PRECTOTCORR  ALLSKY_SFC_PAR_TOT
date                                              
2023-01-01  26.27         0.05               12.97
2023-01-02  25.37         3.67                4.12
2023-01-03  25.03         9.23               10.03
2023-01-04  22.69        12.33                5.72
2023-01-05  20.39         1.21                9.53


 ### `summarize` — quick data overview

 Prints a formatted table with row counts, date range and per-column statistics.

In [11]:
client.summarize(df)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                      Weather Data Profile                                                                       │
│ ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓                                                  │
│ ┃ Property            ┃ Value                                ┃                                                  │
│ ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩                                                  │
│ │ Temporal Resolution │ Daily                                │                                                  │
│ │ Start Date          │ 2023-01-01 00:00:00                  │                                                  │
│ │ End Date            │ 2023-01-15 00:00:00                  │                                                  │
│ │ Data Points         │ 15                                   │                                                  │
│ │ Missing Values      │ 0 / 45                               │                                                  │
│ │ Parameters          │ T2M, PRECTOTCORR, ALLSKY_SFC_PAR_TOT │                                                  │
│ └─────────────────────┴──────────────────────────────────────┘                                                  │
╰───────────────────────────────────────────────── Data Insight ──────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│    Transfer & Cache Performance                                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓                                                                              │
│ ┃ Metric            ┃ Value      ┃                                                                              │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩                                                                              │
│ │ Network Duration  │ 0.00 s     │                                                                              │
│ │ Total Downloaded  │ 0.00 B     │                                                                              │
│ │ Avg Speed         │ 0.00 KiB/s │                                                                              │
│ │ Cache (Initial)   │ 453.00 B   │                                                                              │
│ │ Cache (Increment) │ 0.00 B     │                                                                              │
│ │ Cache (Total)     │ 0.00 B     │                                                                              │
│ └───────────────────┴────────────┘                                                                              │
╰────────────────────────────────────────────────── Performance ──────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│         Request Statistics                                                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓                                                                             │
│ ┃ Metric                 ┃ Value  ┃                                                                             │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩                                                                             │
│ │ Total Logical Requests │ 1      │                                                                             │
│ │ Cache Hits (Full)      │ 1      │                                                                             │
│ │ Network API Calls      │ 0      │                                                                             │
│ │ Cache Hit Rate         │ 100.0% │                                                                             │
│ └────────────────────────┴────────┘                                                                             │
╰────────────────────────────────────────────────── Efficiency ───────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│            NASA POWER Connection Info                                                                           │
│ ┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓                                                                │
│ ┃ Property   ┃ Value                           ┃                                                                │
│ ┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩                                                                │
│ │ User Agent │ aidweather/0.1.0                │                                                                │
│ │ Base URL   │ https://power.larc.nasa.gov/api │                                                                │
│ └────────────┴─────────────────────────────────┘                                                                │
╰──────────────────────────────────────────────── API Connection ─────────────────────────────────────────────────╯

 ### A quick look at the data

 Once you have a DataFrame, standard pandas and matplotlib workflows apply.

In [12]:
# Daily mean temperature and rainfall over the fetched period
print("Mean temperature (°C):", df["T2M"].mean().round(2))
print("Total rainfall (mm)  :", df["PRECTOTCORR"].sum().round(2))

# Growing Degree Days (base 10 °C) — an example of a derived variable
if {"T2M"}.issubset(df.columns):
    df["GDD"] = (df["T2M"] - 10).clip(lower=0)
    print("\nGrowing Degree Days per day:")
    print(df[["T2M", "GDD"]].to_string())

Mean temperature (°C): 23.37
Total rainfall (mm)  : 67.59

Growing Degree Days per day:
              T2M    GDD
date                    
2023-01-01  26.27  16.27
2023-01-02  25.37  15.37
2023-01-03  25.03  15.03
2023-01-04  22.69  12.69
2023-01-05  20.39  10.39
2023-01-06  21.58  11.58
2023-01-07  21.92  11.92
2023-01-08  22.72  12.72
2023-01-09  23.68  13.68
2023-01-10  24.42  14.42
2023-01-11  22.96  12.96
2023-01-12  22.53  12.53
2023-01-13  23.30  13.30
2023-01-14  23.18  13.18
2023-01-15  24.58  14.58


 ---
 ## 4 · Caching

 The client saves each response to a local SQLite database
 (`~/.aidweather_cache/aidweather_cache.db` by default).
 On the next request for the same location and date range, data is read
 from disk instead of the network.

In [13]:
cache_cfg = client.cache_cfg
print("Cache enabled:", cache_cfg.get("enabled"))
print("Cache path   :", cache_cfg.get("path"))

Cache enabled: True
Cache path   : /home/clever/.cache/aidweather


 ### Cold vs. hot request timing

In [14]:
# Clear cache to get a true cold read
db_path = Path(cache_cfg.get("path", ".")) / "aidweather_cache.db"
if db_path.exists():
    db_path.unlink()

client = PowerClient(temporal_api="daily")  # re-init on fresh DB

t0 = time.perf_counter()
df_cold = client.get_point_data(lat=lat, lon=lon, start=start, end=end, params=params)
t_cold = time.perf_counter() - t0

t0 = time.perf_counter()
df_hot = client.get_point_data(lat=lat, lon=lon, start=start, end=end, params=params)
t_hot = time.perf_counter() - t0

print(f"Cold request : {t_cold:.3f} s")
print(f"Hot request  : {t_hot:.3f} s  ({t_cold / t_hot:.0f}× faster)")

Cold request : 0.837 s
Hot request  : 0.003 s  (321× faster)


 ### Partial overlap — only the missing days are fetched

 If you extend the date range beyond what's already cached, the client
 fetches **only the new dates** and merges them with the cached data.

In [15]:
df_ext = client.get_point_data(
    lat=lat,
    lon=lon,
    start="2023-01-10",
    end="2023-01-25",  # overlaps with cached range
    params=params,
)

print("Extended range:", df_ext.index.min().date(), "→", df_ext.index.max().date())
print("Rows returned :", len(df_ext))

Extended range: 2023-01-10 → 2023-01-25
Rows returned : 16


 ---
 ## 5 · Multi-point fetching

 ### `get_transect_data` — 1D spatial transect

 Fetches data for evenly-spaced points along a straight-line path between two
 `GeoCoordinate` endpoints. Each point is resolved in parallel via the standard
 point API, so you can request multiple parameters.

 The minimum point spacing is **0.5° (~55 km)** to match the NASA POWER native
 grid resolution. If the requested density exceeds this, `num_points` is clamped
 automatically and an `INFO` message is logged.

In [ ]:
from aidweather import GeoCoordinate

coord_a = GeoCoordinate.from_decimal(-25.0, -51.16)  # southern end
coord_b = GeoCoordinate.from_decimal(-20.0, -51.16)  # northern end (~555 km)

df_transect = client.get_transect_data(
    start_coord=coord_a,
    end_coord=coord_b,
    start="2023-01-01",
    end="2023-01-05",
    params=["T2M", "PRECTOTCORR"],
    num_points=3,  # 3 evenly-spaced points along the path
    max_workers=3,
)

print("Transect shape:", df_transect.shape)
print("\nUnique locations:")
print(df_transect[["lat", "lon"]].drop_duplicates().reset_index(drop=True))
print()
print(df_transect.head(9))

Transect shape: (15, 6)

Unique locations:
    lat    lon
0 -22.5 -51.16
1 -25.0 -51.16
2 -20.0 -51.16

        date    T2M  PRECTOTCORR   lat    lon     name
0 2023-01-01  27.67         0.69 -22.5 -51.16  Point_2
1 2023-01-02  27.18        18.37 -22.5 -51.16  Point_2
2 2023-01-03  26.24        17.94 -22.5 -51.16  Point_2
3 2023-01-04  24.10        23.11 -22.5 -51.16  Point_2
4 2023-01-05  22.81         2.00 -22.5 -51.16  Point_2
5 2023-01-01  22.74         0.76 -25.0 -51.16  Point_1
6 2023-01-02  21.94         2.97 -25.0 -51.16  Point_1
7 2023-01-03  21.98        18.88 -25.0 -51.16  Point_1
8 2023-01-04  19.40         9.96 -25.0 -51.16  Point_1


 You can also use `spacing_km` instead of `num_points` to control density,
 or use the convenience wrapper `get_transect_data_from_coordinates`:

In [17]:
df_transect_spacing = client.get_transect_data_from_coordinates(
    coord_a=coord_a,
    coord_b=coord_b,
    start="2023-01-01",
    end="2023-01-05",
    params=["T2M"],
    spacing_km=200,  # one point every ~200 km
    max_workers=3,
)
print("Points fetched (spacing_km=200):", df_transect_spacing["lat"].nunique())

Points fetched (spacing_km=200): 4


 ### `get_multi_point_data` — arbitrary list of locations

 Pass a list of dicts with `lat` and `lon` keys (plus optional `name` and
 `elevation`). Returns a combined DataFrame and a list of any failed points.

In [18]:
locations = [
    {"lat": -23.31, "lon": -51.16, "name": "Londrina"},
    {"lat": -15.78, "lon": -47.93, "name": "Brasília"},
    {"lat": -30.03, "lon": -51.23, "name": "Porto Alegre"},
]

df_multi, failed = client.get_multi_point_data(
    points=locations,
    start="2023-01-01",
    end="2023-01-05",
    params=["T2M"],
    max_workers=3,
)

print("Multi-point shape:", df_multi.shape)
if failed:
    print("Failed points:", failed)

mean_temperatures = df_multi.groupby(["name"])["T2M"].mean()

print(mean_temperatures)

Multi-point shape: (15, 5)
name
Brasília        22.014
Londrina        23.950
Porto Alegre    26.196
Name: T2M, dtype: float64


 ---
 ## 6 · Data cleanup — `ensure_date_column`

 Raw datasets don't always have a column called `"date"`.
 `ensure_date_column` searches a list of candidate column names and
 normalises whatever it finds to a plain date string (`YYYY-MM-DD`).
 If no column matches it can fall back to the DataFrame's index.

In [19]:
# --- Scenario A: non-standard column name with timestamp strings ---
raw_a = pd.DataFrame(
    {
        "obs_date": [
            "2023-05-15 08:30:00",
            "2023-05-16 12:45:00",
            "2023-05-17 19:15:00",
        ],
        "T2M": [18.5, 20.2, 17.8],
    }
)

clean_a = ensure_date_column(
    raw_a, name="date", candidates=["obs_date", "measurement_time"]
)
print("Before:")
print(raw_a)
print("\nAfter :")
print(clean_a)
print("dtype :", clean_a["date"].dtype)

Before:
              obs_date   T2M
0  2023-05-15 08:30:00  18.5
1  2023-05-16 12:45:00  20.2
2  2023-05-17 19:15:00  17.8

After :
    T2M       date
0  18.5 2023-05-15
1  20.2 2023-05-16
2  17.8 2023-05-17
dtype : datetime64[us]


In [20]:
# --- Scenario B: date lives in the DatetimeIndex, not in a column ---
raw_b = pd.DataFrame(
    {"RH2M": [65.0, 72.0, 58.0]},
    index=pd.date_range("2023-06-01", periods=3, freq="D"),
)

clean_b = ensure_date_column(raw_b, name="date", index_fallback=True)
print("Before:")
print(raw_b)
print("\nAfter :")
print(clean_b)

Before:
            RH2M
2023-06-01  65.0
2023-06-02  72.0
2023-06-03  58.0

After :
            RH2M       date
2023-06-01  65.0 2023-06-01
2023-06-02  72.0 2023-06-02
2023-06-03  58.0 2023-06-03


 ---
 ## Summary

 | Component | What it does |
 |-----------|--------------|
 | `cfg` / `get_config()` | Read API endpoint URLs and parameter metadata from config |
 | `GeoCoordinate` | Parse, validate and convert geographic coordinates |
 | `normalize_coord_input` | One-call parsing for mixed float/string coordinate tuples |
 | `PowerClient.get_point_data` | Fetch daily or hourly data for one location |
 | `PowerClient.get_multi_point_data` | Fetch for multiple locations in parallel |
 | `PowerClient.get_transect_data` | Fetch a 1D transect between two `GeoCoordinate` endpoints |
 | `PowerClient.get_transect_data_from_coordinates` | Convenience wrapper for transects using two corner coords |
 | `PowerClient.get_regional_data` | Fetch a 0.5° grid within a bounding box (1 param, daily only) |
 | `PowerClient.get_regional_data_from_coordinates` | Regional fetch using two corner `GeoCoordinate` objects |
 | `PowerClient.summarize` | Print a formatted overview of any fetched DataFrame |
 | `ensure_date_column` | Normalise date columns in any DataFrame |

 All fetched data is cached locally. Subsequent calls for the same
 location and date range return immediately from disk.